# Fine-tuning time-series emergence models

This notebook trains and fine-tunes time-series models for detecting seed germination/emergence from short image sequences.

The workflow is intentionally simple:

1. Load preprocessed NumPy datasets.
2. Tune and train TCN/LSTM temporal models with an EfficientNetV2 image backbone.
3. Evaluate trained models on the test split.
4. Fine-tune an existing pretrained model on the selected dataset.

## Imports and project paths

This section imports the training libraries and defines the local project paths. Advanced users can edit the path constants directly when running the notebook on another machine.

In [ ]:
from pathlib import Path

import keras_tuner as kt  # type: ignore
import numpy as np
import tensorflow as tf
from keras import Model
from keras.applications.efficientnet_v2 import EfficientNetV2B0, preprocess_input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, TensorBoard
from keras.layers import (
    LSTM,
    BatchNormalization,
    Dense,
    Flatten,
    Input,
    Lambda,
    TimeDistributed,
)
from keras.models import load_model
from matplotlib import pyplot as plt
from tcn import TCN
from tensorflow import keras  # type: ignore

# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("/home/vasakjakub/sprout-emergence-prediction")
DATA_ROOT = ROOT_PATH / "data"
PREDICTION_MODEL_DIR = ROOT_PATH / "models" / "prediction_model"
NUMPY_DATASET_DIR = DATA_ROOT / "numpy_dataset"
LOG_ROOT = ROOT_PATH / "logs"

print("Repo root:", ROOT_PATH)
print("Prediction model dir:", PREDICTION_MODEL_DIR)
print("NumPy dataset dir:", NUMPY_DATASET_DIR)
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
print(tf.__version__)

## Training settings

These variables control where datasets are loaded from and define the default image/time-series settings used throughout the notebook.

In [ ]:
# Keep legacy string paths for compatibility with older notebook code.
main_dir = str(ROOT_PATH) + "/"
data_path = f"{DATA_ROOT}/"
model_path = f"{PREDICTION_MODEL_DIR}/"
data_images_path = NUMPY_DATASET_DIR

# Image parameters must match the generated NumPy dataset files.
height, width, colors = 75, 75, 3

# Step size used to localize the moment of germination precisely.
slide_step = 1

# Default maximum number of epochs. Early stopping usually ends training sooner.
num_epochs = 300

## Training-curve visualization

Use this helper after model training to compare training and validation loss/accuracy over epochs.

In [ ]:
def visualize_training(model_history: keras.callbacks.History) -> None:
    """Plot loss and accuracy curves for a fitted Keras model.

    Args:
        model_history: Training history returned by ``model.fit``.
    """
    plt.figure(figsize=(30, 10))
    plt.plot(np.array(model_history.history["loss"]), "r--", label="Train loss")
    plt.plot(np.array(model_history.history["accuracy"]), "g--", label="Train accuracy")
    plt.plot(np.array(model_history.history["val_loss"]), "r-", label="Validation loss")
    plt.plot(
        np.array(model_history.history["val_accuracy"]),
        "g-",
        label="Validation accuracy",
    )
    plt.title("Training session's progress over iterations")
    plt.legend(loc="upper right")
    plt.ylabel("Training Progress (Loss/Accuracy)")
    plt.xlabel("Training Epoch")
    plt.ylim(0)
    plt.show()

## Data and model utility functions

These helpers save trained models, load prepared NumPy datasets, evaluate models, and create common callbacks.

In [ ]:
def save_model(dir_name: str, model_name: Model) -> None:
    """Save a trained Keras model under the prediction-model directory.

    Args:
        dir_name: Name of the output directory for the saved model.
        model_name: Fitted Keras model instance to save.
    """
    directory = PREDICTION_MODEL_DIR / dir_name
    directory.mkdir(parents=True, exist_ok=True)

    model_name.save(directory)
    print(f"Model: {dir_name} was saved.")

In [ ]:
def load_data_npy(
    base_path: Path,
    time_step: int,
    size: int,
    sample: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Load image sequences and labels from prepared NumPy files.

    The file naming convention is fixed by the preprocessing pipeline:
    ``data_{height}_{time_step}_{size}_{sample}.npy`` for images and the
    same name with ``_ann`` for annotations.

    Args:
        base_path: Directory containing the generated NumPy dataset files.
        time_step: Start day/time index used in the dataset filename.
        size: Time-window size used in the dataset filename.
        sample: Dataset split name, for example ``train``, ``val``, or ``test``.

    Returns:
        A tuple containing the image sequence array and annotation array.
    """
    train_sample = np.load(base_path / f"data_{height}_{time_step}_{size}_{sample}.npy")
    annotation_sample = np.load(
        base_path / f"data_{height}_{time_step}_{size}_{sample}_ann.npy"
    )

    # Print a quick class-balance check before training or evaluation.
    unique_values, counts = np.unique(annotation_sample, return_counts=True)
    print(
        f"Data has shape: {train_sample.shape} with annotations: {annotation_sample.shape}"
    )
    print(f"Unique annotation values: {unique_values} with counts: {counts}\n")

    return train_sample, annotation_sample

In [ ]:
def evaluate_model(
    time_step: int,
    size: int,
    model_name: Model | str,
    path: Path | None = None,
) -> None:
    """Evaluate a trained or saved model on the test dataset split.

    Args:
        time_step: Start day/time index used in the test dataset filename.
        size: Time-window size used in the test dataset filename.
        model_name: In-memory model to evaluate, or saved model directory name.
        path: Optional directory containing the saved model.

    Raises:
        TypeError: If ``path`` is provided but ``model_name`` is not a string.
    """
    x_test, y_test = load_data_npy(
        base_path=data_images_path, time_step=time_step, sample="test", size=size
    )

    if path is None:
        loaded_model = model_name
    else:
        if not isinstance(model_name, str):
            raise TypeError("model_name must be a string when path is provided.")
        loaded_model = load_model(str(path / model_name))

    evaluation = loaded_model.evaluate(x_test, y_test)  # type: ignore[attr-defined]

    print("Evaluation Loss:", evaluation[0])
    print("Evaluation Accuracy:", evaluation[1])

In [ ]:
def create_callbacks(mode: str) -> EarlyStopping:
    """Create an early-stopping callback for tuning or final training.

    Args:
        mode: Callback mode. Use ``tuner`` for shorter patience during
            hyperparameter search; any other value uses final-training patience.

    Returns:
        Configured Keras ``EarlyStopping`` callback.
    """
    if mode == "tuner":
        return EarlyStopping(
            monitor="val_loss", mode="min", patience=3, restore_best_weights=True
        )
    return EarlyStopping(
        monitor="val_loss", mode="min", patience=5, restore_best_weights=True
    )

## Model-building functions

The model combines an EfficientNetV2 image encoder with a temporal layer. The temporal layer is either TCN or LSTM, selected by the tuner type.

In [ ]:
def tcn_layer_tuner(hp: kt.HyperParameters) -> TCN:
    """Create a tunable TCN layer for temporal sequence modeling.

    Args:
        hp: Keras Tuner hyperparameter object used to sample layer settings.

    Returns:
        Configured ``TCN`` layer.
    """
    nb_filters = hp.Int("nb_filters", min_value=4, max_value=30, step=2)
    kernel_size = hp.Choice("kernel_size", values=[2, 3, 4, 5, 6])
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.05, step=0.01)
    nb_stacks = hp.Int("nb_stacks", min_value=1, max_value=2, step=1)

    return TCN(
        nb_filters=nb_filters,
        kernel_size=kernel_size,
        nb_stacks=nb_stacks,
        dilations=(1, 2, 4),
        padding="causal",
        use_skip_connections=True,
        dropout_rate=dropout_rate,
        return_sequences=False,
        activation="sigmoid",
        kernel_initializer="he_normal",
        use_batch_norm=False,
        use_layer_norm=False,
        use_weight_norm=False,
    )

In [ ]:
def lstm_layer_tuner(hp: kt.HyperParameters) -> LSTM:
    """Create a tunable LSTM layer for temporal sequence modeling.

    Args:
        hp: Keras Tuner hyperparameter object used to sample the unit count.

    Returns:
        Configured ``LSTM`` layer.
    """
    units = hp.Choice("lstm_units", values=[32, 64, 128, 256, 512])
    return LSTM(units, return_sequences=False)

In [ ]:
def define_model_efficientnet_b0(
    hp: kt.HyperParameters,
    time_steps: int,
    h: int,
    w: int,
    c: int,
    tuner_type: str,
) -> Model:
    """Build the EfficientNetV2 + temporal-layer classification model.

    Args:
        hp: Keras Tuner hyperparameter object.
        time_steps: Number of images in each input sequence.
        h: Input image height.
        w: Input image width.
        c: Number of input image channels.
        tuner_type: Temporal layer type, either ``tcn_tuner`` or ``lstm_tuner``.

    Returns:
        Uncompiled Keras model that predicts one binary emergence label.

    Raises:
        ValueError: If ``tuner_type`` is not supported.
    """
    if tuner_type == "tcn_tuner":
        time_layer = tcn_layer_tuner(hp)
    elif tuner_type == "lstm_tuner":
        time_layer = lstm_layer_tuner(hp)
    else:
        raise ValueError("Enter a correct function")

    shape = (time_steps, h, w, c)
    inputs = Input(shape=shape)

    # Apply EfficientNet preprocessing independently to each frame.
    x = TimeDistributed(Lambda(lambda img: preprocess_input(img)))(inputs)  # noqa: PLW0108

    base_cnn = EfficientNetV2B0(
        input_shape=(h, w, c), include_top=False, weights="imagenet"
    )
    # TimeDistributed reuses the same image encoder for every frame in a sequence.
    x = TimeDistributed(base_cnn)(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Flatten())(x)
    x = time_layer(x)
    x = Dense(1, activation="sigmoid")(x)

    return Model(inputs=[inputs], outputs=[x])

In [ ]:
def build_model(
    hp: kt.HyperParameters,
    tuner_type: str,
    time_step: int,
    height: int,
    width: int,
    colors: int,
) -> Model:
    """Build and compile a model for Keras Tuner.

    Args:
        hp: Keras Tuner hyperparameter object.
        tuner_type: Temporal layer type, either ``tcn_tuner`` or ``lstm_tuner``.
        time_step: Number of frames in each model input sequence.
        height: Input image height.
        width: Input image width.
        colors: Number of input image channels.

    Returns:
        Compiled Keras model ready for training.
    """
    model = define_model_efficientnet_b0(
        hp, time_step, height, width, colors, tuner_type
    )
    learning_rate = hp.Choice(
        "learning_rate",
        values=[
            0.0005,
            0.0001,
            0.00001,
            0.00005,
        ],
    )
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer, loss="binary_crossentropy", metrics=["accuracy"])

    return model

In [ ]:
def create_tuner(
    tuner_type: str,
    time_step: int,
    size: int,
    height: int,
    width: int,
    colors: int,
) -> kt.Hyperband | None:
    """Create a Hyperband tuner for a TCN or LSTM model variant.

    Args:
        tuner_type: Temporal layer type, either ``tcn_tuner`` or ``lstm_tuner``.
        time_step: Start day/time index used in naming tuner outputs.
        size: Time-window size used in naming tuner outputs.
        height: Input image height.
        width: Input image width.
        colors: Number of input image channels.

    Returns:
        Configured Keras Tuner ``Hyperband`` instance, or ``None`` for an
        unsupported tuner type.
    """
    if tuner_type not in {"tcn_tuner", "lstm_tuner"}:
        print("Choose a valid temporal layer type.")
        return None

    project = f"keras_{tuner_type}_{time_step}_{size}"

    return kt.Hyperband(
        lambda hp: build_model(hp, tuner_type, time_step, height, width, colors),
        objective="val_accuracy",
        max_epochs=10,
        factor=3,
        directory=str(PREDICTION_MODEL_DIR / "keras_tuner_dir"),
        project_name=project,
        overwrite=True,
    )

# Training workflow

Run the cells below in order. First choose the dataset window, then tune/train the TCN model, and optionally repeat the same process for the LSTM model.

## Select time window and load datasets

`time_step` and `size` must match the filenames created by the preprocessing step. This cell loads train and validation splits.

In [ ]:
# These values select files such as data_75_2_8_train.npy.
time_step = 2
size = 8

# Load training and validation data once so both model variants use the same split.
data_train, annotations_train = load_data_npy(
    base_path=data_images_path, time_step=time_step, sample="train", size=size
)

data_val, annotations_val = load_data_npy(
    base_path=data_images_path, time_step=time_step, sample="val", size=size
)

## TCN model training

The TCN model uses dilated temporal convolutions to summarize the image sequence. Start with tuner search, then rebuild and train the best model.

In [ ]:
# Create a tuner that searches only TCN-specific temporal parameters.
tuner_tcn = create_tuner("tcn_tuner", time_step, size, height, width, colors)

In [ ]:
# Hyperparameter search is intentionally shorter because it trains many candidates.
tuner_tcn.search(  # type: ignore
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=[
        create_callbacks("tuner"),
    ],
    use_multiprocessing=True,
    workers=8,
)

In [ ]:
# Reuse the best hyperparameters found by the tuner for final training.
best_hps_tcn = tuner_tcn.get_best_hyperparameters(num_trials=1)[0]  # type: ignore

best_hyperparameters_tcn = tuner_tcn.get_best_hyperparameters()[0]  # type: ignore
print("Best hyperparameters:", best_hyperparameters_tcn.values)

In [ ]:
callbacks = [
    create_callbacks("train"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),  # type: ignore
    TensorBoard(log_dir=str(LOG_ROOT / "training" / f"tcn_{time_step}_{size}")),
]

# Build a fresh model with the best tuner settings before final training.
model_tcn = tuner_tcn.hypermodel.build(best_hps_tcn)  # type: ignore
model_tcn.summary()
history_tcn = model_tcn.fit(
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=callbacks,
)

In [ ]:
visualize_training(history_tcn)

In [ ]:
save_model(f"ef2_tcn_tuner_{time_step}-{size}", model_tcn)

In [ ]:
evaluate_model(time_step, size, model_tcn)

## LSTM model training

The LSTM model is an alternative temporal baseline. It uses the same EfficientNetV2 frame encoder and only swaps the temporal layer.

In [ ]:
# Create a tuner that searches only LSTM-specific temporal parameters.
tuner_lstm = create_tuner("lstm_tuner", time_step, size, height, width, colors)

In [ ]:
# LSTM tuning is kept short to compare model families quickly.
tuner_lstm.search(  # type: ignore
    data_train,
    annotations_train,
    epochs=10,
    validation_data=(data_val, annotations_val),
    callbacks=[create_callbacks("tuner")],
)

In [ ]:
# Reuse the best LSTM hyperparameters for final model training.
best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]  # type: ignore

best_hyperparameters_lstm = tuner_lstm.get_best_hyperparameters()[0]  # type: ignore
print("Best hyperparameters:", best_hyperparameters_lstm.values)

In [ ]:
callbacks = [
    create_callbacks("train"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),  # type: ignore
    TensorBoard(log_dir=str(LOG_ROOT / "training" / f"lstm_{time_step}_{size}")),
]

# Build a fresh LSTM model using the best tuner settings.
model_lstm = tuner_lstm.hypermodel.build(best_hps_lstm)  # type: ignore
model_lstm.summary()
history_lstm = model_lstm.fit(
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=callbacks,
)

In [ ]:
visualize_training(history_lstm)

In [ ]:
save_model(f"ef2_lstm_tuner_{time_step}-{size}_all_{height}", model_lstm)

In [ ]:
evaluate_model(time_step, size, model_lstm)

# Fine-tuning a pretrained model

Use this section when you already have a saved model and want to adapt it to a new dataset. The model is loaded, compiled with a small learning rate, and trained with checkpoints and TensorBoard logs.

In [ ]:
import datetime

from keras.callbacks import ModelCheckpoint

In [ ]:
# Load the pretrained model that will be adapted to the current dataset.
model = load_model(str(PREDICTION_MODEL_DIR / "ef2_tcn_2-8_org_nab_train_relearned"))

In [ ]:
# Save the best fine-tuning checkpoints separately from the original model.
fine_tune_checkpoint_dir = PREDICTION_MODEL_DIR / "ef2_tcn_2-8_org_nab_train_finetuned"

callbacks = [
    EarlyStopping(
        monitor="val_loss", mode="min", patience=5, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1,  # type: ignore
    ),
    ModelCheckpoint(
        filepath=str(
            fine_tune_checkpoint_dir / "checkpoint-{epoch:02d}-{val_loss:.4f}"
        ),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    TensorBoard(
        log_dir=str(
            LOG_ROOT
            / "fine_tuning"
            / datetime.datetime.now(tz=datetime.UTC).strftime("%Y%m%d-%H%M%S")
        )
    ),
]

# Fine-tuning uses a small learning rate to avoid overwriting useful pretrained features.
model.compile(  # type: ignore
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(  # type: ignore
    data_train,
    annotations_train,
    validation_data=(data_val, annotations_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)